# AWSP++: ILP-Grounded Adaptive Weighted Sampling for VM Placement

**Paper**: *AWSP++: ILP-Grounded Adaptive Weighted Sampling for Fast, Energy-Aware, Fragmentation-Minimizing Virtual Machine Placement in Heterogeneous Cloud Data Centers*

This notebook walks through:
1. Loading and inspecting the dataset
2. Running AWSP++ placement simulation
3. Comparing against PABFD and GRVMP baselines
4. Reproducing the key result figures from the paper
5. Running the k-sensitivity sweep (Fig. 8)

In [ ]:
import sys, os
sys.path.insert(0, '..')   # adjust if running notebook from notebooks/ folder

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from config import *
from dataset import load_dataset, split_trace
from model import AWSSPlusPlus, PABFDBaseline, GRVMPBaseline, get_baseline
from train import run_simulation
from utils import (
    compute_active_hosts, compute_energy_kwh, compute_weighted_slack,
    plot_k_sensitivity, plot_headroom_tradeoff, paired_ttest
)

print('Imports OK')

## 1. Load Dataset

We use synthetic traces that match the statistical distributions reported in the paper (Table III). Real traces can be loaded by following `data/download_data.py` instructions.

In [ ]:
# Load Google Cluster synthetic trace
# Paper: 1,287,453 VMs, 24h window, diurnal + bursty (Section VI-A)
vms, hosts = load_dataset(
    dataset_name='google',
    n_vms=5_000,
    n_hosts=200,
    seed=DEFAULT_SEED,
    use_synthetic=True,
)

train_vms, val_vms, test_vms = split_trace(vms)

print(f'Total VMs:   {len(vms):,}')
print(f'Total hosts: {len(hosts):,}')
print(f'Train: {len(train_vms)} | Val: {len(val_vms)} | Test: {len(test_vms)}')
print(f'\nSample VM demand: {vms[0].demand}')
print(f'Sample host capacity: {hosts[0].capacity}')

In [ ]:
# Visualise arrival distribution (diurnal pattern)
arrival_hours = [v.arrival_time / 3600 for v in vms]
cpu_demands   = [v.demand.get('cpu', 0) for v in vms]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(arrival_hours, bins=48, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Hour of day')
axes[0].set_ylabel('VM arrivals')
axes[0].set_title('VM Arrival Distribution (24h diurnal pattern)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(cpu_demands, bins=40, color='darkorange', alpha=0.8)
axes[1].set_xlabel('CPU demand (normalised fraction)')
axes[1].set_ylabel('Count')
axes[1].set_title('VM CPU Demand Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Run AWSP++ Simulation

Algorithm 1 (placement) + Algorithm 3 (rebalancing) from the paper.

In [ ]:
# Run AWSP++ with k=4 (paper default, Table IV)
model = AWSSPlusPlus(hosts=hosts, k=K_SAMPLE, seed=DEFAULT_SEED)
awsp_summary, awsp_log = run_simulation(model, train_vms + val_vms)

print('AWSP++ Results:')
for k, v in awsp_summary.items():
    print(f'  {k:<30} {v}')

## 3. Compare Against Baselines

In [ ]:
results = {'AWSP++': awsp_summary}

for bl_name in ['PABFD', 'GRVMP']:
    bl = get_baseline(bl_name, hosts, DEFAULT_SEED)
    bl_s, _ = run_simulation(bl, train_vms + val_vms)
    results[bl_name] = bl_s
    print(f'{bl_name}: active_hosts={bl_s["active_hosts"]}  '
          f'energy={bl_s["energy_kwh"]:.2f} kWh  '
          f'SLA={bl_s["sla_violation_rate"]:.3f}%')

In [ ]:
# Comparison bar chart (matches Table V structure)
methods = list(results.keys())
metrics_to_plot = ['active_hosts', 'energy_kwh', 'weighted_slack', 'sla_violation_rate']
labels = ['Active Hosts', 'Energy (kWh)', 'Weighted Slack', 'SLA Violation (%)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#1f77b4', '#d62728', '#2ca02c']

for ax, metric, label in zip(axes, metrics_to_plot, labels):
    vals = [results[m].get(metric, 0) for m in methods]
    bars = ax.bar(methods, vals, color=colors[:len(methods)], alpha=0.85, edgecolor='white')
    ax.set_title(label, fontsize=10)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    # Annotate bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('AWSP++ vs Baselines (Google Cluster Synthetic Trace)', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Active Hosts Over Time (Fig. 5 Reproduction)

In [ ]:
# Re-run to capture tick logs for all methods
m_awsp = AWSSPlusPlus(hosts=hosts, k=K_SAMPLE, seed=DEFAULT_SEED)
_, log_awsp = run_simulation(m_awsp, train_vms + val_vms)

m_pabfd = get_baseline('PABFD', hosts, DEFAULT_SEED)
_, log_pabfd = run_simulation(m_pabfd, train_vms + val_vms)

m_grvmp = get_baseline('GRVMP', hosts, DEFAULT_SEED)
_, log_grvmp = run_simulation(m_grvmp, train_vms + val_vms)

fig, ax = plt.subplots(figsize=(10, 4))
for log, name, color, ls in [
    (log_awsp,  'AWSP++', '#1f77b4', '-'),
    (log_pabfd, 'PABFD',  '#d62728', '--'),
    (log_grvmp, 'GRVMP',  '#2ca02c', ':'),
]:
    if log:
        times = [e['time_s'] / 3600 for e in log]
        hosts_cnt = [e['active_hosts'] for e in log]
        ax.plot(times, hosts_cnt, label=name, color=color, linestyle=ls, linewidth=2)

ax.set_xlabel('Hour of day', fontsize=11)
ax.set_ylabel('Active hosts', fontsize=11)
ax.set_title('Active Hosts over 24h Diurnal Cycle (Google Cluster v3, Fig. 5)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. k-Sensitivity Analysis (Fig. 8)

In [ ]:
# Section VI-J: sweep k in {1, 2, 4, 8, 16}
k_results = []
for k in K_SWEEP:
    m = AWSSPlusPlus(hosts=hosts, k=k, seed=DEFAULT_SEED)
    s, _ = run_simulation(m, train_vms)
    k_results.append({
        'k': k,
        'energy_kwh': s['energy_kwh'],
        'avg_decision_time_ms': s['avg_decision_time_ms'],
        'sla_violation_rate': s['sla_violation_rate'],
    })
    print(f'k={k:2d}  energy={s["energy_kwh"]:.2f} kWh  td={s["avg_decision_time_ms"]:.4f} ms')

# Plot
ks = [r['k'] for r in k_results]
energies = [r['energy_kwh'] for r in k_results]
latencies = [r['avg_decision_time_ms'] for r in k_results]

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(ks, energies,  'o-', color='#1f77b4', label='Energy (kWh)', linewidth=2)
ax2.plot(ks, latencies, 's--', color='#d62728', label='Latency (ms)', linewidth=2)
ax1.set_xlabel('Sample size k', fontsize=11)
ax1.set_ylabel('Energy (kWh)', fontsize=11, color='#1f77b4')
ax2.set_ylabel('Decision latency (ms)', fontsize=11, color='#d62728')
ax1.set_title('k-Sensitivity (Fig. 8) - Knee at k=4', fontsize=11)
ax1.set_xticks(ks)
ax1.grid(True, alpha=0.3)
ax1.axvline(x=4, color='gray', linestyle=':', label='default k=4')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, fontsize=9)
plt.tight_layout()
plt.show()

## 6. Slack Histogram (Fig. 6)

In [ ]:
from config import RESOURCE_WEIGHTS

def get_host_slacks(model_obj):
    slacks = []
    for h in model_obj.host_states.values():
        if h.active:
            ws = sum(RESOURCE_WEIGHTS.get(r, 1/3) * h.slack.get(r, 0) for r in RESOURCES)
            slacks.append(ws)
    return slacks

awsp_s = get_host_slacks(m_awsp)
pabfd_s = get_host_slacks(m_pabfd)

cap_norm = max(max(awsp_s or [1]), max(pabfd_s or [1]))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([v/cap_norm for v in pabfd_s], bins=25, alpha=0.6, label='PABFD', color='#d62728')
ax.hist([v/cap_norm for v in awsp_s],  bins=25, alpha=0.7, label='AWSP++', color='#1f77b4')
ax.set_xlabel('Per-host weighted slack (normalised)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Per-host Weighted-Slack Distribution (Fig. 6)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('AWSP++ concentrates slack on fewer hosts (sharper, leftward distribution).')

## 7. Single-VM Inference Example

In [ ]:
from dataset import VMRequest
from model import score, post_placement_slack, power_of_k_sample
import random

# Create a fresh model with some pre-loaded VMs
fresh_model = AWSSPlusPlus(hosts=hosts[:50], k=4, seed=42)
for vm in train_vms[:200]:
    fresh_model.place_vm(vm)

# New VM request
new_vm = VMRequest(
    vm_id='demo_vm',
    demand={'cpu': 0.30, 'ram': 0.50, 'bw': 0.10},
    lifetime_s=3600.0,
    arrival_time=0.0,
)

# Run scoring manually for transparency
all_h  = list(fresh_model.host_states.values())
active = [h for h in all_h if h.active]
idle   = [h for h in all_h if not h.active]
candidates = fresh_model.bucket_index.eligible_hosts(new_vm, active, idle)
sampled    = random.sample(candidates, min(4, len(candidates)))

print(f'Eligible candidates in bucket: {len(candidates)}')
print(f'Sampled (k=4): {len(sampled)}')
print()

for h in sampled:
    s, ps, var = score(h, new_vm, h.active)
    print(f'  Host {h.host_id[-8:]}  score={s:.4f}  '
          f'post_slack={ps}  active={h.active}')

chosen_host_id = fresh_model.place_vm(new_vm)
print(f'\nChosen host: {chosen_host_id}')

## 8. Summary Table (Table V style)

In [ ]:
summary_data = {
    'Method':    list(results.keys()),
    'Hosts':     [results[m]['active_hosts'] for m in results],
    'Energy(kWh)': [round(results[m]['energy_kwh'], 2) for m in results],
    'Slack':     [round(results[m]['weighted_slack'], 4) for m in results],
    'SLA(%)':    [round(results[m]['sla_violation_rate'], 3) for m in results],
    'Migs':      [results[m]['migration_count'] for m in results],
    'td(ms)':    [round(results[m]['avg_decision_time_ms'], 4) for m in results],
}
df = pd.DataFrame(summary_data).set_index('Method')
print(df.to_string())
print('\nNote: Full Table V uses 5 seeds and 3 real datasets.')